# NeuroGolf Solver Family: Fill / Additive Marking

This notebook is a starter pipeline for the `fill_enclosed_regions` task family.

Workflow:

1. Load task ids from `task_groups/task_type_groups.json`.
2. Inspect the family metadata from `task_type_map.csv`.
3. Train or infer a per-task ONNX model with `train_family_task`.
4. Save `taskNNN.onnx` files.
5. Build `submission.zip` from the generated models.

The generated maps are heuristic solver-routing labels. Validate against visible examples before submitting.

In [1]:
# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None


BATCH, CH, H, W = 1, 10, 30, 30
MODEL_VERSION = "fill-additive-v0.1"


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    return pd.read_csv(path, dtype={"task_id": str})


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups = load_task_groups(groups_path)
    return groups.get(family, [])


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None:
        raise ImportError("onnx is required to build models")


def make_model(nodes, initializers, opset=10):
    require_onnx()
    dt = TensorProto.FLOAT
    inp = helper.make_tensor_value_info("input", dt, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", dt, [BATCH, CH, H, W])
    graph = helper.make_graph(nodes, "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})


def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float32)
    bias = np.full((CH,), -0.5, dtype=np.float32)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = helper.make_tensor("W", dt, [CH, CH, 1, 1], weights.flatten().tolist())
    b = helper.make_tensor("B", dt, [CH], bias.tolist())
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    return {"right": right, "wrong": wrong, "first_wrong": first_wrong}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 4
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 4
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path


def build_family_submission(family, trainer, data_dir, out_dir, fallback_identity=False, validate=False, task_ids_override=None):
    task_ids = list(task_ids_override) if task_ids_override is not None else family_task_ids(family)
    rows = []
    for task_id in task_ids:
        task = load_task(data_dir, task_id)
        model, info = trainer(task)
        if model is None and fallback_identity:
            model = make_identity_model()
            info = {**info, "fallback": "identity"}
        if model is None:
            rows.append({"task_id": task_id, "saved": False, **info})
            continue
        path = save_model(model, out_dir, task_id)
        row = {"task_id": task_id, "saved": True, "path": str(path), **info}
        if validate:
            try:
                row.update({f"visible_{k}": v for k, v in visible_validation_summary(path, task).items() if k != "first_wrong"})
            except Exception as exc:
                row["visible_error"] = repr(exc)
        rows.append(row)
    zip_path = create_submission_zip(out_dir)
    return rows, zip_path

In [2]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()

FAMILY = 'fill_enclosed_regions'
MODEL_VERSION = 'fill-additive-v0.1'
DATA_DIR, BASE_OUT_DIR = default_paths()
OUT_DIR = BASE_OUT_DIR / FAMILY
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)

DATA_DIR = /kaggle/input/competitions/neurogolf-2026
OUT_DIR = /kaggle/working/working_submission/fill_enclosed_regions
MODEL_VERSION = fill-additive-v0.1


## Fill / Additive Marking Version Contract

This notebook should track each solver version explicitly:

- `MODEL_VERSION`: human-readable solver version.
- selected tasks: rows from `task_type_map.csv` where `primary_family == "fill_enclosed_regions"`.
- architecture: ONNX nodes, ops, initializer shapes, file size, parameter count.
- performance: exact-match accuracy on `train`, `test`, `arc-gen`, and all visible examples.
- memory profile: parameter count, static tensor memory, runtime profile memory when `onnxruntime` is available.

The competition score for a correct task is driven by `params + memory_bytes`, so keep both visible.

In [3]:
# ONNX dependency setup for model export.
# Dry-run rule fitting can run without ONNX, but build_family_submission
# must import onnx to create taskNNN.onnx files.
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ['onnx', 'onnxruntime'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing ONNX packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import onnx
import onnxruntime as ort
from onnx import TensorProto, helper

print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing ONNX packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.9 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [4]:
task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()
local_3x3_df = family_df[family_df.candidate_flags.str.contains('local_3x3_consistent', na=False)].copy()
task_ids = local_3x3_df['task_id'].tolist()

print('family:', FAMILY)
print('family tasks:', len(family_df))
print('selected local_3x3 tasks:', len(task_ids))
display(local_3x3_df.head(5))

family: fill_enclosed_regions
family tasks: 59
selected local_3x3 tasks: 8


,task_id,task_num,primary_family,confidence,candidate_flags,n_train,n_test,n_arc_gen,n_examples,shape_relation,...,global_mapping,mapping_conflicts,fixed_geometric_transforms,local_3x3_score,local_3x3_conflicts,local_3x3_samples,input_nonzero_preserved_ratio,added_nonzero_cells,changed_cells,notes
14,task015,15,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,3,1,261,265,same_shape_variable_size,...,"{""0"":0,""1"":1,""2"":2,""6"":6,""8"":8}",1776,NaN,1.0,0,4860,1.0,1776,1776,Same shape; input is mostly preserved while ne...
80,task081,81,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,2,1,261,264,same_shape_variable_size,...,"{""0"":0,""8"":8}",634,NaN,1.0,0,2940,1.0,634,634,Same shape; input is mostly preserved while ne...
94,task095,95,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,2,1,262,265,same_shape_variable_size,...,"{""0"":0,""5"":5}",7368,NaN,1.0,0,4860,1.0,7368,7368,Same shape; input is mostly preserved while ne...
219,task220,220,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,4,1,262,267,same_shape_variable_size,...,"{""0"":0,""2"":2,""3"":3,""8"":8}",4248,NaN,1.0,0,9055,1.0,4248,4248,Same shape; input is mostly preserved while ne...
229,task230,230,fill_enclosed_regions,medium,adds_new_color_preserves_input|local_3x3_consi...,3,1,262,266,same_shape_variable_size,...,"{""0"":0,""5"":5}",3392,NaN,1.0,0,10250,1.0,3392,3392,Same shape; input is mostly preserved while ne...


In [5]:
# Inspect one task quickly.
if task_ids:
    sample_task_id = task_ids[0]
    sample_task = load_task(DATA_DIR, sample_task_id)
    print(sample_task_id, 'examples:', len(all_examples(sample_task)))
    print('first input shape:', grid_shape(sample_task['train'][0]['input']))
    print('first output shape:', grid_shape(sample_task['train'][0]['output']))
    print('first input:', sample_task['train'][0]['input'])
    print('first output:', sample_task['train'][0]['output'])
else:
    print('No tasks currently mapped to this family.')

task015 examples: 265
first input shape: (9, 9)
first output shape: (9, 9)
first input: [[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0]]
first output: [[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 4, 0, 0, 0, 0, 0], [0, 0, 2, 0, 0, 0, 0, 0, 0], [0, 4, 0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 0, 0], [0, 0, 0, 0, 0, 7, 1, 7, 0], [0, 0, 0, 0, 0, 0, 7, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0]]


In [6]:
# Local 3x3 selection table: these are the tasks this notebook version is responsible for.
selection_cols = [
    'task_id',
    'confidence',
    'n_train',
    'n_test',
    'n_arc_gen',
    'shape_relation',
    'input_shape_modes',
    'output_shape_modes',
    'input_color_list',
    'output_color_list',
    'new_output_color_list',
    'candidate_flags',
]
fill_selection = local_3x3_df[selection_cols].reset_index(drop=True)
print('selected local_3x3 fill/additive tasks:', len(fill_selection))
display(fill_selection)

selected local_3x3 fill/additive tasks: 8


,task_id,confidence,n_train,n_test,n_arc_gen,shape_relation,input_shape_modes,output_shape_modes,input_color_list,output_color_list,new_output_color_list,candidate_flags
0,task015,medium,3,1,261,same_shape_variable_size,9x9:265,9x9:265,"[0,1,2,6,8]","[0,1,2,4,6,7,8]","[4,7]",adds_new_color_preserves_input|local_3x3_consi...
1,task081,medium,2,1,261,same_shape_variable_size,7x7:264,7x7:264,"[0,8]","[0,1,8]",[1],adds_new_color_preserves_input|local_3x3_consi...
2,task095,medium,2,1,262,same_shape_variable_size,9x9:265,9x9:265,"[0,5]","[0,1,5]",[1],adds_new_color_preserves_input|local_3x3_consi...
3,task220,medium,4,1,262,same_shape_variable_size,15x15:37;14x14:36;13x13:31;12x12:29;11x11:29,15x15:37;14x14:36;13x13:31;12x12:29;11x11:29,"[0,2,3,8]","[0,1,2,3,4,6,8]","[1,4,6]",adds_new_color_preserves_input|local_3x3_consi...
4,task230,medium,3,1,262,same_shape_variable_size,15x15:149;10x10:117,15x15:149;10x10:117,"[0,5]","[0,1,2,3,4,5]","[1,2,3,4]",adds_new_color_preserves_input|local_3x3_consi...
5,task258,medium,3,1,262,same_shape_variable_size,10x10:52;7x7:49;9x9:45;6x6:43;8x8:39,10x10:52;7x7:49;9x9:45;6x6:43;8x8:39,"[0,1]","[0,1,2]",[2],adds_new_color_preserves_input|local_3x3_consi...
6,task331,medium,2,1,262,same_shape_variable_size,10x10:265,10x10:265,"[0,1]","[0,1,2,6,7,8]","[2,6,7,8]",adds_new_color_preserves_input|local_3x3_consi...
7,task352,medium,3,1,262,same_shape_variable_size,8x9:32;10x10:30;8x8:24;7x7:24;5x6:24,8x9:32;10x10:30;8x8:24;7x7:24;5x6:24,"[0,2,3,4,5,6,7,8,9]","[0,1,2,3,4,5,6,7,8,9]",[1],adds_new_color_preserves_input|local_3x3_consi...


In [7]:
# Symbolic 3x3 CNN trainer for fill/additive marking tasks.
#
# This trains a tiny CNN per task when the task can be represented as:
#   output_color_at_cell = f(3x3 input patch around that cell)
# over the full padded [30, 30] canvas.
#
# Architecture:
#   Conv(3x3 exact patch detectors) -> Relu -> Conv(1x1 detector-to-color logits)
#
# It is symbolic rather than gradient-trained: examples define exact patch rules.

CLEAR = 10
ZERO_HOT = -1


def input_canvas(grid):
    canvas = np.full((H, W), CLEAR, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def output_canvas(grid):
    canvas = np.full((H, W), ZERO_HOT, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def extract_color_patch_key(canvas, row, col):
    vals = []
    for rr in range(row - 1, row + 2):
        for cc in range(col - 1, col + 2):
            if 0 <= rr < H and 0 <= cc < W:
                vals.append(int(canvas[rr, cc]))
            else:
                vals.append(CLEAR)
    return tuple(vals)


def learn_local_patch_rules(task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]

    patch_to_color = {}
    conflicts = 0
    total_positions = 0

    for ex in examples:
        x = input_canvas(ex['input'])
        y = output_canvas(ex['output']).reshape(-1)
        xp = np.pad(x, ((1, 1), (1, 1)), constant_values=CLEAR)
        patch_cols = np.stack([
            xp[0:H, 0:W], xp[0:H, 1:W + 1], xp[0:H, 2:W + 2],
            xp[1:H + 1, 0:W], xp[1:H + 1, 1:W + 1], xp[1:H + 1, 2:W + 2],
            xp[2:H + 2, 0:W], xp[2:H + 2, 1:W + 1], xp[2:H + 2, 2:W + 2],
        ], axis=-1).reshape(-1, 9)

        for patch_arr, color in zip(patch_cols, y):
            patch = tuple(patch_arr.tolist())
            color = int(color)
            total_positions += 1
            prev = patch_to_color.get(patch)
            if prev is None:
                patch_to_color[patch] = color
            elif prev != color:
                conflicts += 1

    return patch_to_color, {
        'total_positions': total_positions,
        'unique_patches': len(patch_to_color),
        'conflicts': conflicts,
        'invalid_targets': 0,
    }


def make_symbolic_patch_cnn(patch_to_color):
    require_onnx()
    dt = TensorProto.FLOAT

    # Sort for deterministic model files.
    rules = sorted(patch_to_color.items(), key=lambda kv: (kv[1], kv[0]))
    n_rules = len(rules)
    if n_rules == 0:
        return None

    # Hidden exact-match detectors. Each key is a 3x3 patch of colors 0..9
    # plus CLEAR=10 for zero-hot cells outside the original input grid.
    # For expected color k: +1 on channel k and -1 on every other channel.
    # For expected CLEAR: -1 on every channel, so any color breaks the match.
    W1 = np.zeros((n_rules, CH, 3, 3), dtype=np.float32)
    B1 = np.zeros((n_rules,), dtype=np.float32)

    for out_idx, (patch, _color) in enumerate(rules):
        ones = 0
        k = 0
        for kr in range(3):
            for kc in range(3):
                expected_color = patch[k]
                if expected_color == CLEAR:
                    W1[out_idx, :, kr, kc] = -1.0
                else:
                    W1[out_idx, :, kr, kc] = -1.0
                    W1[out_idx, int(expected_color), kr, kc] = 1.0
                    ones += 1
                k += 1
        B1[out_idx] = -(ones - 0.5)

    # Detector-to-color layer. ZERO_HOT rules intentionally connect to no output
    # channel, leaving all output logits negative for cells outside output grid.
    W2 = np.zeros((CH, n_rules, 1, 1), dtype=np.float32)
    B2 = np.full((CH,), -0.5, dtype=np.float32)
    for detector_idx, (_patch, color) in enumerate(rules):
        if color >= 0:
            W2[int(color), detector_idx, 0, 0] = 2.0

    initializers = [
        helper.make_tensor('W_patch', dt, list(W1.shape), W1.flatten().tolist()),
        helper.make_tensor('B_patch', dt, list(B1.shape), B1.tolist()),
        helper.make_tensor('W_color', dt, list(W2.shape), W2.flatten().tolist()),
        helper.make_tensor('B_color', dt, list(B2.shape), B2.tolist()),
    ]
    nodes = [
        helper.make_node(
            'Conv',
            ['input', 'W_patch', 'B_patch'],
            ['patch_logits'],
            kernel_shape=[3, 3],
            pads=[1, 1, 1, 1],
        ),
        helper.make_node('Relu', ['patch_logits'], ['patch_hits']),
        helper.make_node(
            'Conv',
            ['patch_hits', 'W_color', 'B_color'],
            ['output'],
            kernel_shape=[1, 1],
        ),
    ]
    return make_model(nodes, initializers, opset=10)


MAX_PATCH_DETECTORS = 2500


def fit_symbolic_patch_rules(task):
    # Learn task-specific 3x3 rules without constructing ONNX. This is safe for
    # dry-runs even when the notebook environment does not have onnx installed.
    patch_to_color, stats = learn_local_patch_rules(task)
    if stats['conflicts']:
        return None, {'ok': False, 'reason': '3x3 local patch conflicts', **stats}
    if stats['unique_patches'] > MAX_PATCH_DETECTORS:
        return None, {'ok': False, 'reason': 'too many 3x3 patch detectors', 'max_detectors': MAX_PATCH_DETECTORS, **stats}

    return patch_to_color, {
        'ok': True,
        'trainer': 'symbolic_3x3_patch_cnn',
        'model_version': MODEL_VERSION,
        'max_detectors': MAX_PATCH_DETECTORS,
        'estimated_params': int(stats['unique_patches'] * (CH * 3 * 3 + 1 + CH) + CH),
        **stats,
    }


def train_family_task(task):
    patch_to_color, info = fit_symbolic_patch_rules(task)
    if not info.get('ok'):
        return None, info

    model = make_symbolic_patch_cnn(patch_to_color)
    if model is None:
        return None, {'ok': False, 'reason': 'no patch rules learned', **info}

    return model, info

In [8]:
# Dry-run rule fitting on the first few tasks without saving.
# This cell does not require ONNX; it reports whether a task is trainable by
# the symbolic CNN rules. Actual model export still requires the onnx package.
dry_rows = []
for task_id in task_ids[:10]:
    task = load_task(DATA_DIR, task_id)
    patch_to_color, info = fit_symbolic_patch_rules(task)
    has_model = False
    export_error = None
    if info.get('ok') and onnx is not None:
        try:
            model = make_symbolic_patch_cnn(patch_to_color)
            has_model = model is not None
        except Exception as exc:
            export_error = repr(exc)
    dry_rows.append({
        'task_id': task_id,
        'would_train': bool(info.get('ok')),
        'onnx_available': onnx is not None,
        'has_model': has_model,
        'export_error': export_error,
        **info,
    })

pd.DataFrame(dry_rows)

,task_id,would_train,onnx_available,has_model,export_error,ok,trainer,model_version,max_detectors,estimated_params,total_positions,unique_patches,conflicts,invalid_targets
0,task015,True,True,True,None,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,46268,238500,458,0,0
1,task081,True,True,True,None,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,26977,237600,267,0,0
2,task095,True,True,True,None,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,4252,238500,42,0,0
3,task220,True,True,True,None,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,9302,240300,92,0,0
4,task230,True,True,True,None,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,4656,239400,46,0,0
5,task258,True,True,True,None,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,21119,239400,209,0,0
6,task331,True,True,True,None,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,8292,238500,82,0,0
7,task352,True,True,True,None,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,68185,239400,675,0,0


In [9]:
# Build one model file for every selected local_3x3 task.
# This notebook version targets only tasks whose visible examples are consistent
# with the symbolic 3x3 additive marking model. No fallback models are emitted.
import shutil

# Clear stale models from earlier wider runs before creating this local_3x3 zip.
for old_model_path in OUT_DIR.glob('task*.onnx'):
    old_model_path.unlink()

rows, zip_path = build_family_submission(
    FAMILY,
    train_family_task,
    DATA_DIR,
    OUT_DIR,
    fallback_identity=False,
    validate=False,
    task_ids_override=task_ids,
)

result_df = pd.DataFrame(rows)
display(result_df)
saved_count = int(result_df.get('saved', pd.Series(dtype=bool)).sum()) if len(result_df) else 0
print('selected local_3x3 tasks:', len(task_ids))
print('models saved:', saved_count)
if len(result_df) and 'trainer' in result_df:
    display(result_df['trainer'].fillna('none').value_counts().rename_axis('trainer').reset_index(name='count'))

# Kaggle looks for /kaggle/working/submission.zip when submitting from a notebook.
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
print('family zip:', zip_path)
print('kaggle submission zip:', submission_zip)

,task_id,saved,path,ok,trainer,model_version,max_detectors,estimated_params,total_positions,unique_patches,conflicts,invalid_targets
0,task015,True,/kaggle/working/working_submission/fill_enclos...,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,46268,238500,458,0,0
1,task081,True,/kaggle/working/working_submission/fill_enclos...,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,26977,237600,267,0,0
2,task095,True,/kaggle/working/working_submission/fill_enclos...,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,4252,238500,42,0,0
3,task220,True,/kaggle/working/working_submission/fill_enclos...,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,9302,240300,92,0,0
4,task230,True,/kaggle/working/working_submission/fill_enclos...,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,4656,239400,46,0,0
5,task258,True,/kaggle/working/working_submission/fill_enclos...,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,21119,239400,209,0,0
6,task331,True,/kaggle/working/working_submission/fill_enclos...,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,8292,238500,82,0,0
7,task352,True,/kaggle/working/working_submission/fill_enclos...,True,symbolic_3x3_patch_cnn,fill-additive-v0.1,2500,68185,239400,675,0,0


selected local_3x3 tasks: 8
models saved: 8


,trainer,count
0,symbolic_3x3_patch_cnn,8


family zip: /kaggle/working/working_submission/fill_enclosed_regions/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [10]:
# Model/version manifest for this notebook run.
run_manifest = {
    'family': FAMILY,
    'model_version': MODEL_VERSION,
    'task_count': len(task_ids),
    'out_dir': str(OUT_DIR),
}
run_manifest

{'family': 'fill_enclosed_regions',
 'model_version': 'fill-additive-v0.1',
 'task_count': 8,
 'out_dir': '/kaggle/working/working_submission/fill_enclosed_regions'}

In [11]:
# Optional: validate saved ONNX models on visible examples.
# This can be slow for large families and requires onnxruntime.
validate_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({
        'task_id': row['task_id'],
        'right': summary['right'],
        'wrong': summary['wrong'],
    })

pd.DataFrame(validate_rows)

,task_id,right,wrong
0,task015,265,0
1,task081,264,0
2,task095,265,0
3,task220,267,0
4,task230,266,0
5,task258,266,0
6,task331,265,0
7,task352,266,0


In [12]:
# Submission helper.
# For a full competition submission, combine models from multiple family
# folders into one directory, then call create_submission_zip(combined_dir).
submission_zip = create_submission_zip(OUT_DIR)
print(submission_zip)

/kaggle/working/working_submission/fill_enclosed_regions/submission.zip


In [13]:
# Architecture, performance, and memory report for saved models.
# This cell expects train_family_task to save one or more ONNX models.
# It reports the metrics the competition cares about: file size, parameter
# count, and memory profile, plus train/test/arc-gen exact-match performance.

report_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    try:
        report = model_report(row['path'], task=task)
        arch = report['architecture']
        mem = report['memory_profile']
        perf = report['performance']
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'file_size_bytes': arch.get('file_size_bytes'),
            'params': arch.get('params'),
            'nodes': arch.get('nodes'),
            'op_counts': json.dumps(arch.get('op_counts', {}), sort_keys=True),
            'static_memory_bytes': mem.get('static_memory_bytes'),
            'runtime_memory_bytes': mem.get('runtime_memory_bytes'),
            'train_right': perf['train']['right'],
            'train_total': perf['train']['total'],
            'train_accuracy': perf['train']['accuracy'],
            'test_right': perf['test']['right'],
            'test_total': perf['test']['total'],
            'test_accuracy': perf['test']['accuracy'],
            'arc_gen_right': perf['arc_gen']['right'],
            'arc_gen_total': perf['arc_gen']['total'],
            'arc_gen_accuracy': perf['arc_gen']['accuracy'],
            'visible_right': perf['visible_all']['right'],
            'visible_total': perf['visible_all']['total'],
            'visible_accuracy': perf['visible_all']['accuracy'],
        })
    except Exception as exc:
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'profile_error': repr(exc),
        })

profile_df = pd.DataFrame(report_rows)
display(profile_df)

,task_id,model_version,file_size_bytes,params,nodes,op_counts,static_memory_bytes,runtime_memory_bytes,train_right,train_total,train_accuracy,test_right,test_total,test_accuracy,arc_gen_right,arc_gen_total,arc_gen_accuracy,visible_right,visible_total,visible_accuracy
0,task015,fill-additive-v0.1,185443,46268,3,"{""Conv"": 2, ""Relu"": 1}",3297600,3333600,3,3,1.0,1,1,1.0,261,261,1.0,265,265,1.0
1,task081,fill-additive-v0.1,108277,26977,3,"{""Conv"": 2, ""Relu"": 1}",1922400,1958400,2,2,1.0,1,1,1.0,261,261,1.0,264,264,1.0
2,task095,fill-additive-v0.1,17372,4252,3,"{""Conv"": 2, ""Relu"": 1}",302400,338400,2,2,1.0,1,1,1.0,262,262,1.0,265,265,1.0
3,task220,fill-additive-v0.1,37574,9302,3,"{""Conv"": 2, ""Relu"": 1}",662400,698400,4,4,1.0,1,1,1.0,262,262,1.0,267,267,1.0
4,task230,fill-additive-v0.1,18990,4656,3,"{""Conv"": 2, ""Relu"": 1}",331200,367200,3,3,1.0,1,1,1.0,262,262,1.0,266,266,1.0
5,task258,fill-additive-v0.1,84845,21119,3,"{""Conv"": 2, ""Relu"": 1}",1504800,1540800,3,3,1.0,1,1,1.0,262,262,1.0,266,266,1.0
6,task331,fill-additive-v0.1,33534,8292,3,"{""Conv"": 2, ""Relu"": 1}",590400,626400,2,2,1.0,1,1,1.0,262,262,1.0,265,265,1.0
7,task352,fill-additive-v0.1,273111,68185,3,"{""Conv"": 2, ""Relu"": 1}",4860000,4896000,3,3,1.0,1,1,1.0,262,262,1.0,266,266,1.0


In [14]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote profile: /kaggle/working/working_submission/fill_enclosed_regions/fill_enclosed_regions_fill-additive-v0.1_profile.csv
wrote manifest: /kaggle/working/working_submission/fill_enclosed_regions/fill_enclosed_regions_fill-additive-v0.1_manifest.json
